# Module 05 — Lecture 3: Spike-Timing-Dependent Plasticity (STDP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_05_networks_plasticity/03_stdp_learning.ipynb)

---

**Hebb's rule (1949):** *"Neurons that fire together, wire together."*  
STDP refines this: the **timing** of pre vs post spikes determines whether synapses strengthen or weaken.

**Learning objectives:**
- Implement the STDP learning rule with eligibility traces on GPU
- Observe Hebbian weight potentiation emerge from correlated input
- Understand soft weight bounds and weight distribution evolution
- Profile the overhead of STDP vs fixed-weight simulation

In [ ]:
!nvidia-smi

## 1. The STDP Rule

For a synapse from pre-neuron $i$ to post-neuron $j$:

$$\Delta w = \begin{cases}
A_+ e^{-\Delta t / \tau_+} & \text{if } \Delta t = t_{post} - t_{pre} > 0 \text{ (pre before post → LTP)} \\
-A_- e^{\Delta t / \tau_-} & \text{if } \Delta t < 0 \text{ (post before pre → LTD)}
\end{cases}$$

**Eligibility trace implementation** (online, no spike history needed):
- `x_pre[i]`: incremented when pre fires, decays with $\tau_+$
- `x_post[j]`: incremented when post fires, decays with $\tau_-$

On each timestep:
1. Decay both traces: `x *= exp(-dt/tau)`
2. On pre-spike: `x_pre[i] += 1` → LTD: `dw -= A_- * x_post[j]`
3. On post-spike: `x_post[j] += 1` → LTP: `dw += A_+ * x_pre[i]`

**Soft weight bounds** prevent weights from saturating at 0 or w_max:
- LTP: `dw = A_+ * x_pre[i] * (w_max - w) / w_max`
- LTD: `dw = -A_- * x_post[j] * (w - w_min) / w_max`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# STDP window: dw vs delta_t
tau_plus  = 20.0  # ms
tau_minus = 20.0  # ms
A_plus    = 0.01
A_minus   = 0.0105  # slightly asymmetric → net depression at random firing

dt_arr = np.linspace(-80, 80, 500)
dw = np.where(
    dt_arr > 0,
     A_plus  * np.exp(-dt_arr / tau_plus),
    -A_minus * np.exp( dt_arr / tau_minus)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(dt_arr, dw * 1000, 'b-', lw=2)
axes[0].axhline(0, color='k', lw=0.8)
axes[0].axvline(0, color='gray', linestyle=':', alpha=0.7)
axes[0].fill_between(dt_arr, dw*1000, 0, where=dw>0, color='steelblue', alpha=0.3, label='LTP (×10³)')
axes[0].fill_between(dt_arr, dw*1000, 0, where=dw<0, color='coral',     alpha=0.3, label='LTD (×10³)')
axes[0].set_xlabel('Δt = t_post − t_pre (ms)', fontsize=12)
axes[0].set_ylabel('Δw (×10⁻³)', fontsize=12)
axes[0].set_title('STDP Learning Window', fontsize=13)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Trace dynamics after a single pre-spike then post-spike
T = 200; dt = 0.1; t = np.arange(int(T/dt)) * dt
t_pre, t_post = 50.0, 70.0

x_pre = np.zeros_like(t); x_post = np.zeros_like(t)
for i in range(1, len(t)):
    x_pre[i]  = x_pre[i-1]  * np.exp(-dt / tau_plus)
    x_post[i] = x_post[i-1] * np.exp(-dt / tau_minus)
    if abs(t[i] - t_pre)  < dt/2: x_pre[i]  += 1.0
    if abs(t[i] - t_post) < dt/2: x_post[i] += 1.0

axes[1].plot(t, x_pre,  'b-',  lw=2, label='x_pre (LTP trace)')
axes[1].plot(t, x_post, 'r--', lw=2, label='x_post (LTD trace)')
axes[1].axvline(t_pre,  color='b', linestyle=':', alpha=0.6, label=f't_pre={t_pre} ms')
axes[1].axvline(t_post, color='r', linestyle=':', alpha=0.6, label=f't_post={t_post} ms')
# Mark LTP update: at t_post, dw = A+ * x_pre[at t_post]
i_post = int(t_post / dt)
axes[1].scatter([t_post], [x_pre[i_post]], s=100, color='g', zorder=5,
                label=f'LTP: Δw=A+·x_pre={A_plus*x_pre[i_post]:.4f}')
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_ylabel('Eligibility trace', fontsize=12)
axes[1].set_title('Pre→Post: LTP Update via Trace', fontsize=13)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stdp_window.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Full STDP Simulation on GPU

We combine the `network_sim.cu` architecture (CSR, conductance-based LIF) with the STDP traces from `stdp.cu`.

**Setup:** 400 neurons (80% E, 20% I), 10% connectivity, heterogeneous input currents. Run for 2000 ms and watch weights evolve.

In [ ]:
%%writefile stdp_sim.cu
/*
 * stdp_sim.cu — Full STDP network simulation
 * 400 excitatory neurons, all-to-all excitatory connectivity with STDP
 * Fixed inhibitory background to maintain ~10 Hz baseline activity
 *
 * Records: weight matrix snapshots at t=0, 500, 1000, 2000 ms
 * Compile: nvcc -O2 -o stdp_sim stdp_sim.cu -lm
 */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ float c_tau_E, c_E_E;
__constant__ int   c_T_ref;
__constant__ float c_tau_plus, c_tau_minus, c_A_plus, c_A_minus;
__constant__ float c_w_max, c_w_min;

// ─── Decay eligibility traces ────────────────────────────────────────────────
__global__ void decay_traces(float* x_pre, float* x_post, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    x_pre[i]  *= expf(-c_dt / c_tau_plus);
    x_post[i] *= expf(-c_dt / c_tau_minus);
}

// ─── LIF update + post-spike LTP ─────────────────────────────────────────────
__global__ void lif_step(
    float* V, float* g_E, int* ref, float* x_pre, float* x_post,
    const int* row_in, const int* col_in,
    float* weights, const int* syn_in_to_out,  // maps incoming edge to weights[] index
    int* fired, const float* I_ext, int N
) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= N) return;

    g_E[j] *= expf(-c_dt / c_tau_E);
    fired[j] = 0;
    if (ref[j] > 0) { ref[j]--; V[j] = c_V_reset; return; }

    float I_syn = -g_E[j] * (V[j] - c_E_E);
    V[j] += c_dt / c_tau_m * (-(V[j] - c_E_L) + c_Rm * (I_ext[j] + I_syn));

    if (V[j] >= c_V_th) {
        V[j] = c_V_reset; ref[j] = c_T_ref; fired[j] = 1;
        x_post[j] += 1.0f;
        // LTP: for each incoming synapse i→j
        for (int k = row_in[j]; k < row_in[j+1]; k++) {
            int i   = col_in[k];
            int idx = syn_in_to_out[k];
            float dw = c_A_plus * x_pre[i] * (c_w_max - weights[idx]) / c_w_max;
            atomicAdd(&weights[idx], dw);
        }
    }
}

// ─── Spike propagation + pre-spike LTD ───────────────────────────────────────
__global__ void propagate_and_ltd(
    const int* fired,
    const int* row_out, const int* col_out,
    float* weights, float* g_E,
    float* x_pre, const float* x_post, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;

    x_pre[i] += 1.0f;

    for (int k = row_out[i]; k < row_out[i+1]; k++) {
        int j = col_out[k];
        atomicAdd(&g_E[j], weights[k]);

        // LTD: pre fires after post → weaken synapse
        float dw = -c_A_minus * x_post[j] * (weights[k] - c_w_min) / c_w_max;
        atomicAdd(&weights[k], dw);
    }
}

// ─── Build outgoing CSR ───────────────────────────────────────────────────────
void build_csr_out(int N, float p,
    int** rp, int** ci, float** wv, int* nnz_out, float w_init)
{
    srand(42);
    int* cnt=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) cnt[i]++;
    *rp=(int*)malloc((N+1)*sizeof(int)); (*rp)[0]=0;
    for(int i=0;i<N;i++) (*rp)[i+1]=(*rp)[i]+cnt[i];
    int nnz=(*rp)[N]; *nnz_out=nnz;
    *ci=(int*)malloc(nnz*sizeof(int));
    *wv=(float*)malloc(nnz*sizeof(float));
    srand(42); int* pos=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) {
            int k=(*rp)[i]+pos[i]++; (*ci)[k]=j; (*wv)[k]=w_init; }
    free(cnt); free(pos);
}

// Build incoming CSR from outgoing and create syn_in_to_out map
void build_csr_in(int N, const int* rp_out, const int* ci_out, int nnz,
    int** rp_in, int** ci_in, int** syn_map)
{
    // Count in-degrees
    int* in_cnt=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int k=rp_out[i];k<rp_out[i+1];k++)
        in_cnt[ci_out[k]]++;
    *rp_in=(int*)malloc((N+1)*sizeof(int)); (*rp_in)[0]=0;
    for(int j=0;j<N;j++) (*rp_in)[j+1]=(*rp_in)[j]+in_cnt[j];
    *ci_in  =(int*)malloc(nnz*sizeof(int));
    *syn_map=(int*)malloc(nnz*sizeof(int));
    int* pos=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int k=rp_out[i];k<rp_out[i+1];k++) {
        int j=ci_out[k];
        int slot=(*rp_in)[j]+pos[j]++;
        (*ci_in)[slot]=i;   // pre-neuron i
        (*syn_map)[slot]=k; // index into weights[]
    }
    free(in_cnt); free(pos);
}

int main(int argc, char** argv)
{
    int   N    = (argc>1) ? atoi(argv[1]) : 400;
    float T_ms = (argc>2) ? atof(argv[2]) : 2000.f;
    float dt   = 0.1f;
    int   T    = (int)(T_ms / dt);

    // LIF params
    float tau_m=20.f, E_L=-65.f, Rm=10.f, V_th=-55.f, V_reset=-70.f;
    int   T_ref=(int)(2.f/dt);
    float tau_E=5.f, E_E=0.f;

    // STDP params
    float tau_plus=20.f, tau_minus=20.f;
    float A_plus=0.005f, A_minus=0.00525f;
    float w_max=0.5f, w_min=0.0f, w_init=0.1f;
    float p_conn=0.2f;

    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,       &dt,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,    &tau_m,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,      &E_L,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,       &Rm,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,     &V_th,     sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset,  &V_reset,  sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,    &T_ref,    sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_E,    &tau_E,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_E,      &E_E,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_plus, &tau_plus, sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_minus,&tau_minus,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_A_plus,   &A_plus,   sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_A_minus,  &A_minus,  sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_max,    &w_max,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_min,    &w_min,    sizeof(float)));

    // Build connectivity
    int *h_rp_out,*h_ci_out, nnz;
    float* h_wv;
    build_csr_out(N, p_conn, &h_rp_out, &h_ci_out, &h_wv, &nnz, w_init);

    int *h_rp_in, *h_ci_in, *h_syn_map;
    build_csr_in(N, h_rp_out, h_ci_out, nnz,
                 &h_rp_in, &h_ci_in, &h_syn_map);

    printf("STDP network: N=%d, nnz=%d, T=%.0f ms\n", N, nnz, T_ms);

    // Device arrays
    float *d_V,*d_gE,*d_Iext,*d_xpre,*d_xpost,*d_wv;
    int   *d_ref,*d_fired;
    int   *d_rp_out,*d_ci_out,*d_rp_in,*d_ci_in,*d_syn_map;

    CUDA_CHECK(cudaMalloc(&d_V,    N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_gE,   N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_Iext, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_xpre, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_xpost,N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_ref,  N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_fired,N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_wv,   nnz*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_rp_out,(N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_ci_out, nnz*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_rp_in, (N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_ci_in,  nnz*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_syn_map,nnz*sizeof(int)));

    // Init
    float* h_V    = (float*)malloc(N*sizeof(float));
    float* h_Iext = (float*)malloc(N*sizeof(float));
    srand(7);
    for(int i=0;i<N;i++) {
        h_V[i]    = E_L + (V_th-E_L)*(float)rand()/RAND_MAX;
        h_Iext[i] = 1.2f + 0.6f*(float)rand()/RAND_MAX;
    }
    CUDA_CHECK(cudaMemcpy(d_V,    h_V,    N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Iext, h_Iext, N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_gE,   0, N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_xpre, 0, N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_xpost,0, N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_ref,  0, N*sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_wv,      h_wv,     nnz*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_rp_out,  h_rp_out,(N+1)*sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_ci_out,  h_ci_out, nnz*sizeof(int),  cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_rp_in,   h_rp_in, (N+1)*sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_ci_in,   h_ci_in,  nnz*sizeof(int),  cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_syn_map, h_syn_map,nnz*sizeof(int),  cudaMemcpyHostToDevice));

    int thr=256, blk=(N+thr-1)/thr;

    // Snapshot times
    int snap_steps[] = {0, (int)(500/dt), (int)(1000/dt), (int)(2000/dt)-1};
    int n_snap = 4;
    int snap_idx = 0;

    FILE* fsnap = fopen("weights_snapshots.txt", "w");
    FILE* fspikes= fopen("spikes_stdp.txt", "w");

    cudaEvent_t t0,t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    int* h_fired = (int*)malloc(N*sizeof(int));

    for(int step=0; step<T; step++) {
        float t_ms = step * dt;

        decay_traces<<<blk,thr>>>(d_xpre, d_xpost, N);
        lif_step<<<blk,thr>>>(d_V, d_gE, d_ref, d_xpre, d_xpost,
                               d_rp_in, d_ci_in, d_wv, d_syn_map,
                               d_fired, d_Iext, N);
        propagate_and_ltd<<<blk,thr>>>(d_fired, d_rp_out, d_ci_out,
                                       d_wv, d_gE, d_xpre, d_xpost, N);

        // Record spikes every 10 steps
        if (step % 10 == 0) {
            CUDA_CHECK(cudaMemcpy(h_fired, d_fired, N*sizeof(int), cudaMemcpyDeviceToHost));
            for(int i=0;i<N;i++) if(h_fired[i]) fprintf(fspikes,"%d %.1f\n", i, t_ms);
        }

        // Weight snapshots
        if (snap_idx < n_snap && step == snap_steps[snap_idx]) {
            CUDA_CHECK(cudaMemcpy(h_wv, d_wv, nnz*sizeof(float), cudaMemcpyDeviceToHost));
            fprintf(fsnap, "# t=%.0f ms\n", t_ms);
            for(int k=0;k<nnz;k++) fprintf(fsnap,"%.6f\n", h_wv[k]);
            snap_idx++;
        }
    }

    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    float sim_ms;
    CUDA_CHECK(cudaEventElapsedTime(&sim_ms, t0, t1));
    printf("GPU time: %.2f ms (%.1fx real-time)\n", sim_ms, T_ms/sim_ms);

    fclose(fsnap); fclose(fspikes);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_V); cudaFree(d_gE); cudaFree(d_Iext);
    cudaFree(d_xpre); cudaFree(d_xpost);
    cudaFree(d_ref); cudaFree(d_fired);
    cudaFree(d_wv); cudaFree(d_rp_out); cudaFree(d_ci_out);
    cudaFree(d_rp_in); cudaFree(d_ci_in); cudaFree(d_syn_map);
    free(h_V); free(h_Iext); free(h_fired);
    free(h_rp_out); free(h_ci_out); free(h_wv);
    free(h_rp_in); free(h_ci_in); free(h_syn_map);
    return 0;
}

In [ ]:
!nvcc -O2 -o stdp_sim stdp_sim.cu -lm && ./stdp_sim 400 2000

## 3. Visualizing Weight Evolution

STDP drives the weight distribution away from the uniform initial state. Correlated pre-post pairs get potentiated; anti-correlated pairs get depressed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parse weight snapshots
snapshots = {}
with open('weights_snapshots.txt') as f:
    current_t, current_w = None, []
    for line in f:
        line = line.strip()
        if line.startswith('#'):
            if current_t is not None:
                snapshots[current_t] = np.array(current_w)
            current_t = float(line.split('=')[1].split()[0])
            current_w = []
        else:
            current_w.append(float(line))
    if current_t is not None:
        snapshots[current_t] = np.array(current_w)

times = sorted(snapshots.keys())
print(f"Snapshots at: {times} ms")
print(f"Synapses: {len(snapshots[times[0]])}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
colors = ['steelblue', 'orange', 'green', 'crimson']

for idx, (t, ax) in enumerate(zip(times, axes)):
    w = snapshots[t]
    ax.hist(w, bins=50, color=colors[idx], alpha=0.8, edgecolor='white', lw=0.3)
    ax.axvline(w.mean(), color='k', linestyle='--', lw=1.5, label=f'Mean={w.mean():.3f}')
    ax.set_title(f't = {t:.0f} ms', fontsize=13)
    ax.set_xlabel('Synaptic weight w', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 0.5)

plt.suptitle('STDP: Weight Distribution Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('stdp_weight_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\nWeight statistics over time:")
for t in times:
    w = snapshots[t]
    print(f"  t={t:6.0f} ms: mean={w.mean():.4f}, std={w.std():.4f}, "
          f"bimodal_ratio={((w<0.05).sum()+(w>0.45).sum())/len(w)*100:.1f}%")

In [ ]:
# Raster plot of spike activity
import numpy as np
import matplotlib.pyplot as plt

spike_data = np.loadtxt('spikes_stdp.txt')
n_spikes = len(spike_data)
print(f"Total spikes: {n_spikes}")

if n_spikes > 0:
    neuron_ids = spike_data[:, 0].astype(int)
    spike_times = spike_data[:, 1]

    N = 400
    T_ms = 2000.0
    mean_rate = n_spikes / (N * T_ms / 1000)
    print(f"Mean firing rate: {mean_rate:.1f} Hz")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                    gridspec_kw={'height_ratios': [3, 1]})

    # Raster
    sample = np.where(neuron_ids < 100)[0]  # first 100 neurons
    ax1.scatter(spike_times[sample], neuron_ids[sample],
                s=1, c='black', alpha=0.6, rasterized=True)
    ax1.set_xlim(0, T_ms)
    ax1.set_ylim(-1, 100)
    ax1.set_ylabel('Neuron ID', fontsize=12)
    ax1.set_title('STDP Network: Spike Raster (first 100 neurons)', fontsize=13)
    ax1.axvline(500,  color='r', linestyle=':', alpha=0.5)
    ax1.axvline(1000, color='r', linestyle=':', alpha=0.5)
    ax1.text(510, 95, 'Snapshot', color='r', fontsize=9)

    # Population firing rate
    bin_ms = 20
    bins = np.arange(0, T_ms + bin_ms, bin_ms)
    counts, _ = np.histogram(spike_times, bins=bins)
    rate_hz = counts / (N * bin_ms / 1000)
    ax2.plot(bins[:-1] + bin_ms/2, rate_hz, 'b-', lw=1.5)
    ax2.fill_between(bins[:-1] + bin_ms/2, rate_hz, alpha=0.3)
    ax2.set_xlim(0, T_ms)
    ax2.set_xlabel('Time (ms)', fontsize=12)
    ax2.set_ylabel('Pop. rate (Hz)', fontsize=12)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('stdp_raster.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Why Bimodal Weights?

Under STDP with soft bounds, the weight distribution becomes **bimodal** — synapses cluster near 0 (silent) or w_max (strong). This is a well-known property of STDP:

- The soft bounds create a **positive feedback**: strong synapses cause correlated firing → more LTP → stronger
- The depression term (A_minus > A_plus) creates a competition: not all synapses can be strong
- Result: **winner-take-all** — a subset of pre-neurons dominates the drive to each post-neuron

This is thought to underlie the **selectivity** of cortical neurons: a cell responds strongly to a specific input pattern.

## 5. GPU Implementation Notes

**Atomicity issue:** Both `lif_step` (LTP) and `propagate_and_ltd` (LTD) write to `weights[]` using `atomicAdd`. Because these two kernels run sequentially (not concurrently), there is **no race condition between LTP and LTD**. Within each kernel, two threads could update the same weight (unlikely for sparse networks) — atomicAdd handles this correctly.

**Performance tradeoff:** STDP adds:
- 2 trace arrays (x_pre, x_post) — extra memory bandwidth
- atomicAdd to weights — serialization on hot synapses
- Incoming CSR (rp_in, ci_in) — double the connectivity storage

For large N, the STDP overhead is typically 2-3× slower than fixed-weight simulation.

## Summary

| Concept | Implementation |
|---------|---------------|
| STDP window | Eligibility traces x_pre, x_post (no spike history buffer) |
| LTP on post-spike | Iterate incoming CSR, dw = A+ × x_pre[i] × soft_bound |
| LTD on pre-spike | Iterate outgoing CSR, dw = −A− × x_post[j] × soft_bound |
| Thread safety | atomicAdd to weights (sparse → few conflicts) |
| Bimodal outcome | Soft bounds + competition → selective strengthening |

**Next:** Exercise 05 — implement BCM (Bienenstock-Cooper-Munro) plasticity, a rate-based alternative to STDP with a sliding threshold.